In [1]:
from openai import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from dotenv import load_dotenv
import tiktoken
import chromadb
import os

load_dotenv()

client_llm = OpenAI(
    api_key=os.getenv('GROQ_TOKEN'),
    base_url=os.getenv('BASE_URL_GROQ')
)

emb_model = OpenAIEmbeddingFunction(
    api_base=os.getenv('LOCAL_URL'),
    model_name=os.getenv('LOCAL_MODEL'),
    api_key=os.getenv('LOCAL_TOKEN')
)

decoder = tiktoken.encoding_for_model('text-embedding-3-small')


In [2]:
articles = [
    {"id": "art-01", "headline": "SpaceX Launches New Starship Rocket",
     "topic": "Science", "year": 2024,
     "keywords": ["space", "rocket", "nasa", "exploration"]},
    {"id": "art-02", "headline": "Bitcoin Reaches All-Time High as Investors Rush In",
     "topic": "Business", "year": 2024,
     "keywords": ["crypto", "bitcoin", "finance", "investment"]},
    {"id": "art-03", "headline": "Champions League Final: Real Madrid vs Manchester City",
     "topic": "Sport", "year": 2023,
     "keywords": ["football", "soccer", "champions league", "madrid"]},
    {"id": "art-04", "headline": "New AI Model Outperforms Humans in Medical Diagnosis",
     "topic": "Tech", "year": 2024,
     "keywords": ["ai", "healthcare", "machine learning", "diagnosis"]},
    {"id": "art-05", "headline": "Global Leaders Meet to Discuss Climate Change",
     "topic": "Science", "year": 2023,
     "keywords": ["climate", "environment", "policy", "emissions"]},
    {"id": "art-06", "headline": "Apple Announces Revolutionary New iPhone",
     "topic": "Tech", "year": 2024,
     "keywords": ["apple", "iphone", "smartphone", "technology"]},
    {"id": "art-07", "headline": "Stock Markets Plunge Amid Recession Fears",
     "topic": "Business", "year": 2023,
     "keywords": ["stocks", "economy", "recession", "wall street"]},
    {"id": "art-08", "headline": "Olympics 2024: USA Wins Gold in Swimming",
     "topic": "Sport", "year": 2024,
     "keywords": ["olympics", "swimming", "usa", "gold medal"]},
    {"id": "art-09", "headline": "Scientists Discover New Species in Amazon Rainforest",
     "topic": "Science", "year": 2023,
     "keywords": ["biology", "amazon", "species", "nature"]},
    {"id": "art-10", "headline": "Tesla Unveils Autonomous Robot for Home Use",
     "topic": "Tech", "year": 2024,
     "keywords": ["tesla", "robot", "automation", "ai"]},
]

In [3]:
# Часть 1
## Задача 1.1

client_db = chromadb.PersistentClient('~/artipy/vibe_code_lifestyle/knowledge_lib/chromadb')

client_db.delete_collection(name='news_articles')

collection = client_db.create_collection(
    name='news_articles',
    embedding_function=emb_model
)

client_db.list_collections()

[Collection(name=news_articles)]

In [4]:
## Задача 1.2

def create_article_text(article):
    return f'''Headline: {article['headline']}
Topic: {article['topic']}
Keywords: {article['keywords']}'''

total_tokens = sum(
    len(decoder.encode(create_article_text(article))) for article in articles
)

cost = 0.00002 * total_tokens/1000

print(f'''Total tokens: {total_tokens}
Cost: ${cost:.6}''')

Total tokens: 324
Cost: $6.48e-06


In [ ]:
## Задача 1.3

ids = [article['id'] for article in articles]
documents = [article['headline'] for article in articles]
metadatas = [{'topic':article['topic'], 'year':article['year']} for article in articles]

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

first_10 = collection.peek()

print(f'Документов в коллекции: {collection.count()}')
print(f'ids: {first_10['ids']}')

Документов в коллекции: 10
ids: ['art-01', 'art-02', 'art-03', 'art-04', 'art-05', 'art-06', 'art-07', 'art-08', 'art-09', 'art-10']


In [6]:
# Часть 2
## Задача 2.1

collection = client_db.get_collection(
    name = 'news_articles',
    embedding_function = emb_model
)

queries = [
    "artificial intelligence and robotics",
    "financial crisis and investment",
    "olympic games and athletic achievements",
]

for query in queries:
    print(f'Запрос: "{query}"')
    results = collection.query(
        query_texts=query,
        n_results=3
    )
    
    for i, doc in enumerate(results['documents'][0]):
        print(f'    {i+1}. {doc} (dist={round(results['distances'][0][i], 4)})')

Запрос: "artificial intelligence and robotics"
    1. Tesla Unveils Autonomous Robot for Home Use (dist=0.5361)
    2. New AI Model Outperforms Humans in Medical Diagnosis (dist=0.82)
    3. SpaceX Launches New Starship Rocket (dist=1.1222)
Запрос: "financial crisis and investment"


    1. Stock Markets Plunge Amid Recession Fears (dist=0.6994)
    2. Bitcoin Reaches All-Time High as Investors Rush In (dist=0.8423)
    3. Global Leaders Meet to Discuss Climate Change (dist=1.0354)
Запрос: "olympic games and athletic achievements"
    1. Olympics 2024: USA Wins Gold in Swimming (dist=0.6658)
    2. Apple Announces Revolutionary New iPhone (dist=1.1024)
    3. Global Leaders Meet to Discuss Climate Change (dist=1.1197)


In [44]:
## Задача 2.2
reference_ids = ['art-02', 'art-08']

reference_texts = collection.get(
    ids=reference_ids
)['documents']

results = collection.query(
    query_texts=reference_texts,
    n_results = 3
)

for i, id in enumerate(reference_ids):
    print(f'Результат для запроса {i+1} (По статье {id}):')
    for j, doc in enumerate(results['documents'][i]):
        print(f'''    {j+1}. {doc} (dist={round(results['distances'][i][j] ,4)})''')

Результат для запроса 1 (По статье art-02):
    1. Bitcoin Reaches All-Time High as Investors Rush In (dist=0.0)
    2. Stock Markets Plunge Amid Recession Fears (dist=0.9664)
    3. Apple Announces Revolutionary New iPhone (dist=0.977)
Результат для запроса 2 (По статье art-08):
    1. Olympics 2024: USA Wins Gold in Swimming (dist=0.0)
    2. Global Leaders Meet to Discuss Climate Change (dist=1.1394)
    3. Apple Announces Revolutionary New iPhone (dist=1.1448)


In [8]:
# Задача 3
## Задаа 3.1

print(collection.get(ids=['art-03']))

collection.update(
    ids=['art-03'],
    metadatas=[{'topic':'Sport', 'year':2023, 'featured':True}]
    )

print(collection.get(ids=['art-03']))


{'ids': ['art-03'], 'embeddings': None, 'documents': ['Champions League Final: Real Madrid vs Manchester City'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'year': 2023, 'topic': 'Sport'}]}
{'ids': ['art-03'], 'embeddings': None, 'documents': ['Champions League Final: Real Madrid vs Manchester City'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'featured': True, 'year': 2023, 'topic': 'Sport'}]}


In [14]:
new_article = [{
    "id": "art-11",
    "headline": "Ethereum Surges 30% Following Network Upgrade",
    "topic": "Business",
    "year": 2024,
    "keywords": ["ethereum", "crypto", "blockchain", "investment"]
}]

ids = [article['id'] for article in new_article]
documents = [article['headline'] for article in new_article]
metadatas = [{'topic':article['topic'], 'year':article['year']} for article in new_article]

collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print(f'''Документов после upsert: {collection.count()}
Метаданные art-03: {collection.get(ids=['art-03'])['metadatas']}''')

Документов после upsert: 11
Метаданные art-03: [{'topic': 'Sport', 'year': 2023, 'featured': True}]


In [15]:
## Задача 3.2
collection.delete(ids=['art-11'])
print(f'''Документов после удаления: {collection.count()}''')

Документов после удаления: 10


In [27]:
# Часть 4
## Задача 4.1
print('=== Только Tech ===')
query_text = 'technology and innovation'
result = collection.query(
    query_texts=[query_text],
    n_results=3,
    where = {'topic':"Tech"}
    )

for i, ids in enumerate(result['ids'][0]):
    print(f'    {i+1}. {result['documents'][0][i]}')

print()

print('=== Только 2023 год ===')
query_text = 'science discoveries'
result = collection.query(
    query_texts=[query_text],
    n_results=3,
    where = {'year':2023}
    )

for i, ids in enumerate(result['ids'][0]):
    print(f'    {i+1}. {result['documents'][0][i]}')

print()

print('=== Исключая Business ===')
query_text = 'sports championships'
result = collection.query(
    query_texts=[query_text],
    n_results=3,
    where = {'topic':{'$ne':'Business'}}
    )

for i, ids in enumerate(result['ids'][0]):
    print(f'    {i+1}. {result['documents'][0][i]}')

=== Только Tech ===
    1. Tesla Unveils Autonomous Robot for Home Use
    2. Apple Announces Revolutionary New iPhone
    3. New AI Model Outperforms Humans in Medical Diagnosis

=== Только 2023 год ===
    1. Scientists Discover New Species in Amazon Rainforest
    2. Global Leaders Meet to Discuss Climate Change
    3. Stock Markets Plunge Amid Recession Fears

=== Исключая Business ===
    1. Olympics 2024: USA Wins Gold in Swimming
    2. Champions League Final: Real Madrid vs Manchester City
    3. Global Leaders Meet to Discuss Climate Change


In [31]:
## Задача 4.2

print('=== Tech И 2024 год ===')

query_text = 'cutting edge technology'

result = collection.query(
    query_texts=[query_text],
    where={
        '$and':[
            {
                'topic':'Tech'
            },
            {
                'year':2024
            }
        ]
    }
)

print(f'Найдено: {len(result['ids'][0])} результатов')

print('\n=== Science ИЛИ Sport ===')

query_text = 'global events'

result = collection.query(
    query_texts=[query_text],
    where={
        '$or':[
            {
                'topic':'Science'
            },
            {
                'topic':'Sport'
            }
        ]
    }
)

print(f'Найдено: {len(result['ids'][0])} результатов')

=== Tech И 2024 год ===
Найдено: 3 результатов

=== Science ИЛИ Sport ===
Найдено: 5 результатов


In [41]:
# Часть 5
## Задача 5.1

reference_id = ['art-02']
reference_text = collection.get(ids=reference_id)['documents']
reference_topic = 'Business'

result = collection.query(
    query_texts=reference_text,
    where={'topic':reference_topic}
)

print(f'Эталонная статья: "{reference_text[0]}"\nТема: "{reference_topic}"\n')
rank = 1
print(f'Рекомендации (только {reference_topic})')
for i, id in enumerate(result['ids'][0]):
    if id == reference_id[0]:
        continue
    else:
        print(f'    {rank}. {result['documents'][0][i]} (dist={round(result['distances'][0][i], 4)})')
        rank +=1 

Эталонная статья: "Bitcoin Reaches All-Time High as Investors Rush In"
Тема: "Business"

Рекомендации (только Business)
    1. Stock Markets Plunge Amid Recession Fears (dist=0.9664)


In [43]:
client_db.delete_collection('news_articles')

collection = client_db.create_collection(
    name='news_articles',
    embedding_function=emb_model
)

batch_size = 5
for i in range(0, len(articles), batch_size):
    batch = articles[i:i + batch_size]

    ids = [article['id'] for article in batch]
    documents = [article['headline'] for article in batch]
    metadatas = [{'topic':article['topic'], 'year':article['year']} for article in batch]

    collection.upsert(
        ids=ids,
        documents=documents,
        metadatas=metadatas
    )
    print(f'Загружен батч {i // batch_size + 1}, документов в коллекции: {collection.count()}')

print(f'\nИтого документов: {collection.count()}')

Загружен батч 1, документов в коллекции: 5
Загружен батч 2, документов в коллекции: 10

Итого документов: 10
